### Import Libraries

In [86]:
import json
import numpy as np
import pandas as pd
from openai import OpenAI
from gitsource import GithubRepositoryDataReader
from gitsource import chunk_documents
from embedder import Embedder
from minsearch import Index
from minsearch import VectorSearch
from evaluation_utils import *
from rag_helper import *

In [87]:
embedder = Embedder()

query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)

In [88]:
print(v[0])

-0.02058203437252893


In [89]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [90]:
doc = next(
    d for d in documents
    if d["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md"
)

In [91]:
doc_vector = embedder.encode(doc["content"])

In [92]:
similarity = v.dot(doc_vector)

print(similarity)

0.36107027225589694


In [93]:
from gitsource import chunk_documents
import numpy as np
chunks = chunk_documents(
    documents,
    size=2000,
    step=1000
)

In [94]:
X = embedder.encode_batch(
    [chunk["content"] for chunk in chunks]
)

In [95]:
scores = X.dot(v)

In [96]:
idx = np.argmax(scores)

chunks[idx]["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [97]:
from minsearch import VectorSearch
vs = VectorSearch(
    keyword_fields=["filename"],
)

vs.fit(
    vectors=X,
    payload=chunks
)

In [98]:
query = "What metric do we use to evaluate a search engine?"
query_vector = embedder.encode(query)

In [99]:
results = vs.search(
    query_vector=query_vector,
    num_results=5
)
results[0]["filename"]

'04-evaluation/lessons/05-search-metrics.md'

In [100]:
from minsearch import Index
text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
text_index.fit(chunks)

In [101]:
query = "How do I store vectors in PostgreSQL?"

In [102]:
query_vector = embedder.encode(query)
vector_results = vs.search(
    query_vector=query_vector,
    num_results=5
)

text_results = text_index.search(
    query=query,
    num_results=5
)

In [103]:
print("VECTOR")
for r in vector_results:
    print(r["filename"])

print("TEXT")
for r in text_results:
    print(r["filename"])

VECTOR
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
TEXT
02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md


In [104]:
vector_files = {r["filename"] for r in vector_results}
text_files = {r["filename"] for r in text_results}
print(vector_files - text_files)

{'02-vector-search/lessons/08-pgvector.md'}


In [105]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])

            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)

    return [docs[key] for key in ranked[:num_results]]

In [106]:
query = "How do I give the model access to tools?"

In [107]:
query_vector = embedder.encode(query)
vector_results = vs.search(
    query_vector=query_vector,
    num_results=5
)
text_results = text_index.search(
    query=query,
    num_results=5
)

In [108]:
results = rrf([vector_results, text_results])
results[0]["filename"]

'01-agentic-rag/lessons/13-function-calling.md'

### **Q1**

In [109]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [110]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

## **Q2**

In [83]:
def text_search(query, num_results=5):
    return text_index.search(
        query=query,
        num_results=num_results
    )

def vector_search(query, num_results=5):
    query_vector = embedder.encode(query)
    return vs.search(
        query_vector=query_vector,
        num_results=num_results
    )

def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)

    return rrf([text_results, vector_results], k=k)

In [111]:
import pandas as pd
ground_truth = pd.read_csv("ground-truth.csv").to_dict(orient="records")
len(ground_truth)

360

In [84]:
q = ground_truth[0]["question"]
print(q)

What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?


In [85]:
results = text_search(q)
results[0]["filename"]

'01-agentic-rag/lessons/03-rag.md'

Q3

In [112]:
results = vector_search(q)
results[0]["filename"]

'01-agentic-rag/lessons/01-intro.md'

Q4

In [115]:
def compute_relevance(search_function, ground_truth, k=5):
    relevance_total = []
    for row in ground_truth:
        results = search_function(row["question"], num_results=k)
        relevance = [
            1 if r["filename"] == row["filename"] else 0
            for r in results
        ]
        relevance_total.append(relevance)
    return relevance_total

In [116]:
def hit_rate(relevance_total):
    hits = 0
    for rel in relevance_total:
        if 1 in rel:
            hits += 1
    return hits / len(relevance_total)

In [117]:
def mrr(relevance_total):
    total = 0
    for rel in relevance_total:
        for rank, value in enumerate(rel):
            if value == 1:
                total += 1 / (rank + 1)
                break
    return total / len(relevance_total)

In [118]:
def evaluate(search_function):
    relevance = compute_relevance(search_function, ground_truth)
    return {
        "hit_rate": hit_rate(relevance),
        "mrr": mrr(relevance),
    }

In [119]:
evaluate(text_search)

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

In [120]:
evaluate(vector_search)

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}

In [121]:
for k in [1, 50, 100, 200]:
    def hybrid_k(query, num_results=5):
        return hybrid_search(query, k=k)
    result = evaluate(hybrid_k)
    print(f"k={k}")
    print(result)
    print()

k=1
{'hit_rate': 0.8388888888888889, 'mrr': 0.6481944444444449}

k=50
{'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}

k=100
{'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}

k=200
{'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}

